# 🚀 Starter Notebook: PyTorch Basics + Your First Training Run

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aminashehzadi393/llm-engineering/blob/main/notebooks/01_pytorch_basics_and_first_training_run.ipynb)

Part of the [LLM Engineering learning roadmap](https://github.com/aminashehzadi393/llm-engineering) — Phases 1 & 2.

## What you'll learn
1. **Tensors** — the basic data structure behind all of deep learning
2. **Autograd** — how PyTorch computes gradients automatically (the engine of training)
3. **Your first training run** — teach a tiny neural network a pattern from data

No setup needed — just run each cell **top to bottom**. Works on free Colab; CPU is totally fine for this.

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## Part 1 — Tensors

A tensor is just a multi-dimensional array. Images, text embeddings, model weights — in deep learning, *everything* is a tensor.

In [ ]:
a = torch.tensor([1, 2, 3])
b = torch.zeros(2, 3)
c = torch.randn(2, 3)  # random numbers from a normal distribution

print("a:", a)
print("b shape:", b.shape)
print("c:\n", c)

# Basic operations
x = torch.tensor([[1., 2.], [3., 4.]])
print("\nx * 2 =\n", x * 2)
print("matrix multiply x @ x.T =\n", x @ x.T)

## Part 2 — Autograd (the magic behind training)

`requires_grad=True` tells PyTorch: *"track every operation on this tensor, so you can tell me its gradient later."*

Training a model = computing gradients, then nudging weights in the direction that reduces the error. Autograd does the calculus for you.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x + 1   # y = x² + 2x + 1 → dy/dx = 2x + 2 → at x=3, that's 8

y.backward()  # compute the gradient
print("dy/dx at x=3:", x.grad.item())  # expect 8.0

## Part 3 — Your first training run 🌱

Let's make a tiny neural network learn a pattern from data: **y = x²** (plus a little noise, like real-world data).

First, create the dataset and take a look at it.

In [ ]:
import matplotlib.pyplot as plt

torch.manual_seed(42)

N = 200
X = torch.rand(N, 1) * 4 - 2          # x values between -2 and 2
y = X ** 2 + 0.3 * torch.randn(N, 1)  # y = x² + noise

plt.scatter(X.numpy(), y.numpy(), alpha=0.5, s=10)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Our data — can a tiny net learn the curve?")
plt.show()

### The model

A small neural network: input → 32 neurons (ReLU) → output. Only ~100 parameters — tiny compared to an LLM's billions, but the *training mechanics are exactly the same*.

In [ ]:
import torch.nn as nn

class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)

model = TinyNet().to(device)
print(model)
print("Total parameters:", sum(p.numel() for p in model.parameters()))

### The training loop — the 5 steps of all deep learning

Every neural network ever trained — including GPT — repeats this loop:

1. **Forward pass** — make a prediction
2. **Loss** — measure how wrong the prediction was
3. **Backward pass** — compute gradients (autograd!)
4. **Optimizer step** — nudge every weight a little downhill
5. **Repeat** — hundreds or thousands of times

In [ ]:
X, y = X.to(device), y.to(device)

criterion = nn.MSELoss()                        # mean squared error = our "how wrong" measure
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

losses = []
epochs = 800
for epoch in range(1, epochs + 1):
    optimizer.zero_grad()      # 1. clear old gradients
    pred = model(X)            # 2. forward pass
    loss = criterion(pred, y)  # 3. how wrong are we?
    loss.backward()            # 4. compute gradients
    optimizer.step()           # 5. nudge weights downhill

    losses.append(loss.item())
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | loss = {loss.item():.4f}")

### Did it learn? 👀

Left: the loss curve going down = learning happening. Right: the red line is what the network predicts — compare it with the dashed true curve.

In [ ]:
model.eval()
with torch.no_grad():
    X_test = torch.linspace(-2, 2, 100).unsqueeze(1).to(device)
    y_pred = model(X_test).cpu()

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Loss going down = learning 📉")

plt.subplot(1, 2, 2)
plt.scatter(X.cpu().numpy(), y.cpu().numpy(), alpha=0.3, s=10, label="data")
plt.plot(X_test.cpu().numpy(), y_pred.numpy(), color="red", linewidth=2, label="model prediction")
plt.plot(X_test.cpu().numpy(), (X_test.cpu() ** 2).numpy(), "k--", alpha=0.5, label="true y = x²")
plt.legend()
plt.title("The tiny net learned the curve! 🎉")

plt.tight_layout()
plt.show()

## 🎓 What you just learned

- **Tensors** are the data format of deep learning
- **Autograd** computes gradients automatically — no manual calculus
- **Training** is one loop repeated: predict → measure error → gradients → update weights
- An LLM trains with *the same loop* — just billions of parameters, trillions of words, and weeks on GPUs

## ✏️ Exercises (try them!)
1. Change the hidden layer from 32 to 4 neurons. Does it still learn the curve? Why / why not?
2. Set `lr=0.5` in the optimizer. What happens to the loss? (This is why learning rate matters.)
3. Change the data to `y = torch.sin(X * 2)` and retrain. Can the same tiny net learn a wave?

## ➡️ Next in the roadmap
- **Notebook 02** — fine-tuning: take a small pre-trained model and adapt it with LoRA
- **Phase 2** — how transformers & attention work (the architecture inside every LLM)

*Tick the "PyTorch fundamentals" box in the README when this clicks. ✅*